---

## SummarizationMiddleware 옵션별 테스트

``SummarizationMiddleware`` 는 대화 기록이 ``trigger`` 임계값을 넘으면 요약 LLM을 호출하고,
``keep`` 정책에 따라 최근 컨텍스트만 남깁니다.

- before_model

**참고:** [Built-in Middleware — Summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

아래 각 섹션은 ``feature/MiddlewareSummarization.py`` 의 ``MiddlewareSummarizationAgent`` 로
옵션 하나씩 바꿔 동작을 확인합니다.

**옵션 요약:**

| 옵션 | 형식 예시 | 역할 |
|:---|:---|:---|
| ``model`` (``summary_model``) | ``"gpt-4o-mini"`` | 요약 생성에 쓸 LLM |
| ``trigger`` | ``("messages", 6)``, ``("tokens", 4000)``, ``("fraction", 0.8)`` | 요약을 **시작**하는 임계값 (복수 지정 가능) |
| ``keep`` | ``("messages", 4)``, ``("tokens", 500)``, ``("fraction", 0.3)`` | 요약 **후** 남길 최근 컨텍스트 |
| ``token_counter`` | 커스텀 callable | 토큰 수 계산 방식 |
| ``summary_prompt`` | 프롬프트 문자열 | 요약 LLM에 넘길 템플릿 (``{messages}`` 자리표시) |
| ``trim_tokens_to_summarize`` | ``4000`` / ``None`` | 요약 호출 전 메시지 트리밍 상한 (앞에 메시지를 그냥 잘라서 4000자로 줄임) |

**``trigger`` / ``keep`` 의 세 가지 단위**

| 단위 | 예시 | 기준 |
|:---|:---|:---|
| ``messages`` | ``("messages", 20)`` | 메시지 **개수**  / 요약본 제외|
| ``tokens`` | ``("tokens", 4000)`` | 토큰 **절대값** (모델 정보 불필요) |
| ``fraction`` | ``("fraction", 0.8)`` | 모델 ``max_input_tokens`` 의 **비율** (프로필 필요 → §12 참고) |

요약이 실행되면 메시지에 ``Here is a summary of the conversation to date:`` 로 시작하는
``HumanMessage`` 가 추가됩니다. ``messages_contain_summary()`` 로 확인합니다.

In [6]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

from feature.MiddlewareSummarization import (
    MiddlewareSummarizationAgent,
    build_demo_messages,
    messages_contain_summary,
)

THREAD = {"configurable": {"thread_id": "summarization-test"}}

### 1. 기본 설정 — 짧은 대화 (요약 없음)

기본 ``trigger=("tokens", 4000)`` / ``keep=("messages", 20)``.
메시지가 적으면 요약이 **실행되지 않아야** 합니다.

In [7]:
agent_default = MiddlewareSummarizationAgent()

result = agent_default.invoke(
    inputs={"messages": [HumanMessage(content="What's the weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "summ-default"}),
)

assert result is not None
assert not messages_contain_summary(result["messages"]), "짧은 대화에서는 요약이 없어야 합니다"
print("✓ 기본 설정 — 요약 미실행 확인")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_3Dj8ma3YU80GQ0hnNJPc6PI4)
 Call ID: call_3Dj8ma3YU80GQ0hnNJPc6PI4
  Args:
    city: Seoul

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!
✓ 기본 설정 — 요약 미실행 확인


### 2. ``trigger=("messages", 6)`` — 메시지 개수 임계값

메시지가 6개 이상이면 요약이 트리거됩니다.
6쌍(12개) Human/AI 메시지를 넣어 요약 실행을 확인합니다.

In [5]:
agent_msg_trigger = MiddlewareSummarizationAgent(
    trigger=("messages", 6),
    keep=("messages", 4),
)

long_history = build_demo_messages(6)
long_history.append(HumanMessage(content="Summarize what we discussed so far."))

result = agent_msg_trigger.invoke(
    inputs={"messages": long_history},
    config=RunnableConfig(configurable={"thread_id": "summ-trigger-messages"}),
)

assert messages_contain_summary(result["messages"]), "메시지 수 trigger — 요약 HumanMessage 기대"
print(f"✓ trigger=(messages, 6) — 메시지 수: {len(result['messages'])}")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to obtain explanations about various topics (Topic 1 through Topic 5) in a sequential manner.

## SUMMARY
The user requested information on five different topics, each time asking for a brief explanation. The responses provided cover the user's requests in a general and concise manner, but the specific content of the explanations for each topic was not detailed in the conversation.

## ARTIFACTS
None

## NEXT STEPS
The user should proceed to ask for an explanation of Topic 5, and the AI can provide a short explanation as requested.
================================== Ai Message ==================================

Topic 5 is interesting.

### 3. ``trigger=("tokens", 80)`` — 토큰 수 임계값

토큰 카운트가 80 이상이면 요약합니다. 긴 텍스트 메시지로 임계값을 넘깁니다.

In [8]:
agent_token_trigger = MiddlewareSummarizationAgent(
    trigger=("tokens", 80),
    keep=("messages", 3),
)

padding = "LangGraph middleware summarization test. " * 30
token_history = build_demo_messages(
    3,
    user_template="Question {i}: " + padding + "{i}",
    ai_template="Answer {i}: " + padding + "{i}",
)
token_history.append(HumanMessage(content="What were the main topics?"))

result = agent_token_trigger.invoke(
    inputs={"messages": token_history},
    config=RunnableConfig(configurable={"thread_id": "summ-trigger-tokens"}),
)

assert messages_contain_summary(result["messages"]), "토큰 trigger — 요약 HumanMessage 기대"
print("✓ trigger=(tokens, 80) — 요약 실행 확인")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to conduct "LangGraph middleware summarization tests" and receive answers aligned with the tests.

## SUMMARY
The user has asked two questions about the "LangGraph middleware summarization test," indicating a focus on this specific topic in multiple entries. The AI provided consistent responses for each question, reiterating the phrase "LangGraph middleware summarization test" without variation.

## ARTIFACTS
None

## NEXT STEPS
None
================================ Human Message =================================

Question 3: LangGraph middleware summarization test. LangGraph middleware summarization test. LangGraph middleware summariz

### 4. ``trigger`` 복수 조건 — OR 조건

``[("messages", 20), ("tokens", 50)]`` 처럼 **하나라도** 넘으면 요약합니다.
여기서는 토큰 조건만 낮게 잡아 트리거합니다.

In [9]:
agent_multi_trigger = MiddlewareSummarizationAgent(
    trigger=[("messages", 20), ("tokens", 50)],
    keep=("messages", 2),
)

short_pad = "token trigger list test. " * 8
multi_history = build_demo_messages(
    2,
    user_template=short_pad + "Q{i}",
    ai_template=short_pad + "A{i}",
)
multi_history.append(HumanMessage(content="Continue."))

result = agent_multi_trigger.invoke(
    inputs={"messages": multi_history},
    config=RunnableConfig(configurable={"thread_id": "summ-trigger-list"}),
)

assert messages_contain_summary(result["messages"]), "복수 trigger — 요약 HumanMessage 기대"
print("✓ trigger list — OR 조건 요약 확인")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is conducting a test related to a "token trigger list."

## SUMMARY
The conversation consists of repeated testing phrases by both the user and the AI, primarily focusing on the "token trigger list" with multiple instances of "token trigger list test." There are sequential queries labeled Q1, A1, Q2 without any additional context or outcomes provided.

## ARTIFACTS
None

## NEXT STEPS
None
================================== Ai Message ==================================

token trigger list test. token trigger list test. token trigger list test. token trigger list test. token trigger list test. token trigger list test. token trigger list test. token trigge

### 5. ``keep=("messages", 4)`` — 요약 후 최근 4개 메시지 유지

요약 후 보존된 메시지 + 요약 HumanMessage 합계가 ``keep + 1`` 수준으로 줄어듭니다.

In [10]:
agent_keep_msg = MiddlewareSummarizationAgent(
    trigger=("messages", 8),
    keep=("messages", 4),
)

keep_history = build_demo_messages(5)
keep_history.append(HumanMessage(content="Final question."))

result = agent_keep_msg.invoke(
    inputs={"messages": keep_history},
    config=RunnableConfig(configurable={"thread_id": "summ-keep-messages"}),
)

msg_count = len(result["messages"])
assert messages_contain_summary(result["messages"])
assert msg_count <= 6, f"keep=(messages,4) — 요약+보존분 합 {msg_count}개 (기대 ≤6)"
print(f"✓ keep=(messages, 4) — 최종 메시지 수: {msg_count}")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to request information on various topics in a sequential manner, specifically seeking explanations for topic 1, topic 2, topic 3, and topic 4.

## SUMMARY
The conversation consists of the user asking the AI to explain four different topics in order:
1. Topic 1 - the AI provided a short explanation.
2. Topic 2 - the AI provided a short explanation.
3. Topic 3 - the AI provided a short explanation.
4. Topic 4 - the user requested information but the AI's response is not included in the conversation history.

## ARTIFACTS
None

## NEXT STEPS
Provide an explanation for topic 4 as requested by the user.
================================== Ai

### 6. ``keep=("tokens", 120)`` — 토큰 기준 보존

요약 후 최근 메시지를 토큰 예산(120) 안에 맞춥니다.

In [11]:
from langchain_core.messages.utils import count_tokens_approximately

agent_keep_tokens = MiddlewareSummarizationAgent(
    trigger=("messages", 8),
    keep=("tokens", 120),
)

keep_token_history = build_demo_messages(5)
keep_token_history.append(HumanMessage(content="Wrap up."))

result = agent_keep_tokens.invoke(
    inputs={"messages": keep_token_history},
    config=RunnableConfig(configurable={"thread_id": "summ-keep-tokens"}),
)

assert messages_contain_summary(result["messages"])
approx_tokens = count_tokens_approximately(result["messages"])
print(f"✓ keep=(tokens, 120) — 요약 후 대략 {approx_tokens} tokens, {len(result['messages'])} messages")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about specific topics in a sequential manner, starting with "topic 1" and then moving to "topic 2."

## SUMMARY
The user requested information on two distinct topics sequentially. In Turn 1, the AI provided a short explanation about "topic 1." In Turn 2, the user asked for information about "topic 2," indicating a continued interest in exploring related subjects.

## ARTIFACTS
None

## NEXT STEPS
Respond to the user's request for information about "topic 2."
================================== Ai Message ==================================

Topic 2 is interesting. Here is a short explanation.
================================ Human M

### 7. ``summary_prompt`` — 커스텀 요약 프롬프트

요약 LLM에 넘기는 프롬프트를 바꿉니다. ``{messages}`` 자리에 대화 기록이 들어갑니다.

In [ ]:
CUSTOM_MARKER = "[CUSTOM_SUMMARY]"
custom_prompt = (
    "Summarize the conversation below in one short paragraph. "
    f"Start your answer with {CUSTOM_MARKER}\n\n"
    "Messages:\n{messages}"
)

agent_custom_prompt = MiddlewareSummarizationAgent(
    trigger=("messages", 6),
    keep=("messages", 2),
    summary_prompt=custom_prompt,
)

prompt_history = build_demo_messages(4)
prompt_history.append(HumanMessage(content="Go on."))

result = agent_custom_prompt.invoke(
    inputs={"messages": prompt_history},
    config=RunnableConfig(configurable={"thread_id": "summ-custom-prompt"}),
)

summary_text = next(
    m.content
    for m in result["messages"]
    if messages_contain_summary([m])
)
assert CUSTOM_MARKER in summary_text, "커스텀 summary_prompt 마커가 요약 본문에 포함되어야 합니다"
print("✓ summary_prompt — 커스텀 마커 확인:", summary_text[:120], "...")

### 8. ``token_counter`` — 커스텀 토큰 카운터

항상 높은 토큰 수를 반환하는 카운터로 ``trigger=("tokens", 100)`` 를 쉽게 넘깁니다.

In [ ]:
def always_high_token_counter(messages):
    return 10_000

agent_fake_counter = MiddlewareSummarizationAgent(
    trigger=("tokens", 100),
    keep=("messages", 2),
    token_counter=always_high_token_counter,
)

result = agent_fake_counter.invoke(
    inputs={"messages": [HumanMessage(content="Hi")]},
    config=RunnableConfig(configurable={"thread_id": "summ-fake-counter"}),
)

assert messages_contain_summary(result["messages"]), "커스텀 token_counter — 즉시 요약 기대"
print("✓ token_counter — 강제 트리거 요약 확인")

### 9. ``trim_tokens_to_summarize`` — 요약 입력 트리밍

요약 LLM 호출 전에 메시지를 ``trim_tokens_to_summarize`` 토큰 이하로 자릅니다.
``None`` 이면 트리밍을 건너뜁니다.

In [4]:
agent_trim = MiddlewareSummarizationAgent(
    trigger=("messages", 6),
    keep=("messages", 2),
    trim_tokens_to_summarize=200,
)

trim_history = build_demo_messages(
    4,
    user_template="Long user message {i}. " + ("x " * 200) + "{i}",
    ai_template="Long ai reply {i}. " + ("y " * 200) + "{i}",
)
trim_history.append(HumanMessage(content="Done?"))

result = agent_trim.invoke(
    inputs={"messages": trim_history},
    config=RunnableConfig(configurable={"thread_id": "summ-trim"}),
)

assert messages_contain_summary(result["messages"])
print("✓ trim_tokens_to_summarize=200 — 트리밍 후 요약 실행 확인")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is not explicitly stated, but it appears to involve discussing or analyzing a lengthy message or concept, indicated by the extensive content.

## SUMMARY
None

## ARTIFACTS
None

## NEXT STEPS
None
================================== Ai Message ==================================

Long ai reply 4. y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y y

### 10. ``summary_model`` — 요약 전용 모델

에이전트 본체 LLM과 **별도** 모델로 요약을 생성합니다.
(동일 모델을 써도 API는 분리 호출됩니다.)

In [12]:
from util.chat_model_enums import LangChainChatModel

agent_sep_model = MiddlewareSummarizationAgent(
    model=LangChainChatModel.OPENAI_GPT_4O_MINI,
    summary_model=LangChainChatModel.OPENAI_GPT_4O_MINI,
    trigger=("messages", 6),
    keep=("messages", 2),
)

sep_history = build_demo_messages(4)
sep_history.append(HumanMessage(content="Continue please."))

result = agent_sep_model.invoke(
    inputs={"messages": sep_history},
    config=RunnableConfig(configurable={"thread_id": "summ-sep-model"}),
)

assert messages_contain_summary(result["messages"])
print("✓ summary_model — 별도 요약 모델로 요약 생성 확인")


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is asking for explanations on multiple topics, seeking concise information about each.

## SUMMARY
The user requested information on four distinct topics, labeled as topic 1, topic 2, topic 3, and topic 4. The AI provided brief explanations for the first three topics, indicating that each topic is interesting. The details of these explanations were not specified in the conversation.

## ARTIFACTS
None

## NEXT STEPS
Provide a short explanation for topic 4 as requested by the user.
================================== Ai Message ==================================

Topic 4 is interesting. Here is a short explanation.
================================ Human M

### 11. 도구 호출 + 요약 미들웨어

``get_weather`` 도구가 붙은 에이전트에서도 요약 미들웨어가 동작합니다.

In [13]:
agent_with_tool = MiddlewareSummarizationAgent(
    trigger=("messages", 6),
    keep=("messages", 4),
)

result = agent_with_tool.invoke(
    inputs={"messages": [HumanMessage(content="What's the weather in Seoul?")]},
    config=RunnableConfig(configurable={"thread_id": "summ-with-tool"}),
)

last = result["messages"][-1]
assert not messages_contain_summary([last]) or len(result["messages"]) > 1
print("✓ 도구 호출 — 응답:", getattr(last, "content", last)[:200])


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_KHn0fg028uX6e62WOQMwMpnm)
 Call ID: call_KHn0fg028uX6e62WOQMwMpnm
  Args:
    city: Seoul

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

It's sunny in Seoul!

🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

The weather in Seoul is sunny!
✓ 도구 호출 — 응답: The weather in Seoul is sunny!


### 12. (선택) ``trigger=("fraction", 0.5)`` / ``keep=("fraction", 0.3)`` — **비율(%) 기준**

앞선 테스트(2~6)는 ``("messages", N)`` 또는 ``("tokens", N)`` 처럼 **고정 숫자**로 임계값을 잡았습니다.
``fraction`` 은 그와 달리 **모델 컨텍스트 창 크기 대비 비율**로 임계값을 잡습니다.

| 옵션 | 의미 | 계산 예 (``max_input_tokens=128_000`` 일 때) |
|:---|:---|:---|
| ``trigger=("fraction", 0.5)`` | 대화 토큰이 **입력 한도의 50%**에 도달하면 요약 시작 | ``128_000 × 0.5 = 64_000`` 토큰 |
| ``keep=("fraction", 0.3)`` | 요약 **후** 최근 기록을 **입력 한도의 30%** 분량만 남김 | ``128_000 × 0.3 ≈ 38_400`` 토큰 |

**왜 ``max_input_tokens``(모델 프로필)가 필요한가?**

- ``("tokens", 4000)`` → 숫자를 직접 쓰므로 모델 정보 없이도 동작
- ``("fraction", 0.5)`` → “이 모델 입력 한도의 절반”이라 **한도 절대값**을 알아야 함
- LangChain은 요약 LLM(``summary_model``)의 ``model.profile["max_input_tokens"]`` 에서 그 값을 읽음
- ``init_chat_model("gpt-4o-mini")`` 처럼 **프로필이 등록된 모델**이면 자동으로 채워짐
- 프로필이 없으면 비율을 계산할 수 없어 **생성 시점에** ``ValueError`` 발생

**언제 쓰면 좋은가?**

- gpt-4o / claude 등 **모델을 바꿔도** “입력 한도의 80%에서 요약”처럼 **동일한 정책**을 유지하고 싶을 때
- 모델마다 ``max_input_tokens`` 가 다르므로, 고정 ``("tokens", 4000)`` 보다 **모델에 맞게 스케일**됨

**프로필이 없을 때 대안**

1. ``("tokens", N)`` / ``("messages", N)`` 처럼 **절대값** 옵션 사용 (권장)
2. 또는 모델 생성 시 직접 지정: ``ChatOpenAI(..., profile={"max_input_tokens": 128000})``

아래 셀은 ``trigger=("fraction", 0.01)`` — 입력 한도의 **1%**만 넘어도 요약을 시도합니다.
프로필이 없으면 ``except`` 에서 스킵 메시지가 출력됩니다.

In [14]:
try:
    agent_fraction = MiddlewareSummarizationAgent(
        trigger=("fraction", 0.01),
        keep=("fraction", 0.5),
    )
    frac_history = build_demo_messages(3)
    frac_history.append(HumanMessage(content="Check fraction trigger."))
    result = agent_fraction.invoke(
        inputs={"messages": frac_history},
        config=RunnableConfig(configurable={"thread_id": "summ-fraction"}),
    )
    print(
        "✓ fraction trigger — 요약 여부:",
        messages_contain_summary(result["messages"]),
    )
except ValueError as e:
    print("⊘ fraction 옵션 스킵 (모델 프로필 없음):", e)


🔄 Node: SummarizationMiddleware.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

It seems you want to check a specific condition or aspect related to fractions. Could you provide more context or clarify what you mean by "fraction trigger"?
✓ fraction trigger — 요약 여부: False
